# Лабораторная работа 3–4: практическая часть

Ноутбук читает исходные файлы из `../raw/dataset0304/`. Начинаем с проверки структуры и качества данных, затем строим метрики потока и симуляцию Монте-Карло.

## 1. Обзор набора данных

В таблицах ниже показаны реальные записи из `bd.db` и CSV. Для прогноза необходимо отдельно выбрать одну команду и проверить её историю завершения.


In [ ]:
from pathlib import Path
import sqlite3
import csv
from collections import Counter
from datetime import datetime
import random
import math

ROOT = Path.cwd()
DATA = ROOT.parent / "raw" / "dataset0304"
if not (DATA / "bd.db").exists():
    # Позволяет запускать ноутбук из корня проекта или из папки src.
    DATA = ROOT / "raw" / "dataset0304"
conn = sqlite3.connect(DATA / "bd.db")
conn.row_factory = sqlite3.Row


### Таблица: задачи по командам

| Команда | Всего задач | Завершено | Не завершено | Первая дата Done | Последняя дата Done |
|---|---|---|---|---|---|
| Web PHP Team | 140 | 115 | 25 | 2024-01-11 | 2026-01-26 |
| Backend API & DB Team | 120 | 101 | 19 | 2024-01-13 | 2026-02-01 |
| Mobile iOS Swift Team | 120 | 102 | 18 | 2024-01-07 | 2026-01-21 |
| Mobile Android Kotlin Team | 120 | 97 | 23 | 2024-01-19 | 2025-12-29 |

### Таблица: текущие статусы

| Текущий статус | Задач |
|---|---|
| Done | 415 |
| Backlog | 38 |
| Selected for Development | 22 |
| In Progress | 13 |
| Blocked | 12 |

### Таблица: история переходов

| Статус в истории | Переходов |
|---|---|
| Done | 863 |
| Backlog | 500 |
| Selected for Development | 462 |
| In Progress | 440 |
| Code Review | 427 |
| QA | 427 |
| Reopened | 21 |
| Blocked | 12 |

### Таблица: файл задач

| CSV-файл | Строк | Поля |
|---|---|---|
| issues.csv | 500 | issue_id, issue_key, project_id, team_id, epic_id, assignee_id, issue_type, priority, summary, created_at, resolved_at, story_points, current_status, sprint_id |

In [ ]:
# Состав набора и число задач по командам
teams = list(conn.execute("SELECT team_id, team_name FROM teams ORDER BY team_id"))
issues = list(conn.execute("SELECT * FROM issues"))
for team in teams:
    team_issues = [i for i in issues if i["team_id"] == team["team_id"]]
    done = [i for i in team_issues if i["resolved_at"]]
    print(f'{team["team_name"]:28} всего: {len(team_issues):3} | завершено: {len(done):3} | не завершено: {len(team_issues)-len(done):3}')

print("\nТекущие статусы:")
for status, count in Counter(i["current_status"] for i in issues).most_common():
    print(f"{status:28} {count:3}")


## 2. Выбор команды и подготовка throughput

На первом проходе выбрана **Backend API & DB Team**. Throughput — число задач, перешедших в Done в каждый календарный день. Дни без завершённых задач оставляем с нулём: иначе скорость будет завышена.


In [ ]:
TEAM_NAME = "Backend API & DB Team"
TEAM_ID = next(t["team_id"] for t in teams if t["team_name"] == TEAM_NAME)
done_issues = [i for i in issues if i["team_id"] == TEAM_ID and i["resolved_at"]]

# Для первой версии используем дату resolved_at; ниже проверим её по истории статусов.
daily = Counter(i["resolved_at"][:10] for i in done_issues)
first_day = min(daily)
last_day = max(daily)
start = datetime.strptime(first_day, "%Y-%m-%d").date()
end = datetime.strptime(last_day, "%Y-%m-%d").date()
throughput = []
day = start
while day <= end:
    throughput.append(daily.get(day.isoformat(), 0))
    day = day.fromordinal(day.toordinal() + 1)

print(f"Команда: {TEAM_NAME}")
print(f"Завершено задач: {len(done_issues)}")
print(f"Период: {first_day} — {last_day} ({len(throughput)} календарных дней)")
print(f"Средний throughput: {sum(throughput)/len(throughput):.3f} задач/день")
print(f"Медианный throughput: {sorted(throughput)[len(throughput)//2]:.0f} задач/день")


## 3. Монте-Карло для завершения бэклога

Следующая ячейка моделирует число календарных дней до закрытия 30 задач, случайно выбирая дневной throughput из истории команды с возвращением. Цель можно изменить в `REMAINING_ISSUES`.


In [ ]:
REMAINING_ISSUES = 30
N_SIMULATIONS = 10_000
rng = random.Random(42)
durations = []
for _ in range(N_SIMULATIONS):
    remaining = REMAINING_ISSUES
    days = 0
    while remaining > 0:
        remaining -= rng.choice(throughput)
        days += 1
    durations.append(days)

def percentile(values, p):
    values = sorted(values)
    index = (len(values) - 1) * p / 100
    lower = math.floor(index)
    upper = math.ceil(index)
    return values[lower] + (values[upper] - values[lower]) * (index - lower)

print(f"Прогноз для {REMAINING_ISSUES} задач (календарные дни), {N_SIMULATIONS:,} симуляций")
for p in (50, 85, 95):
    print(f"P{p}: {percentile(durations, p):.1f} дней")


## Примечание к интерпретации

Это первичный прогноз по одной команде. В следующем этапе следует сверить `resolved_at` с переходами в статус Done из `issue_status_history`, проверить пропуски/дубликаты, построить график ежедневного throughput и оценить стабильность процесса. Если команда или правило включения задач меняются, прогноз необходимо пересчитать.
